In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01010
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.636143093983
Gradient descend method:  None
RUN  0 , total integrated cost =  23532.636143093983
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  140 0.4500000000000001 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  0 , total integrated cost =  10019.968518582271
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  0 , total integrated cost =  33290.05146687772
Improved over  0  iterations in  0.0  seconds by  0.0  percent.


In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  705.4029734175052
RUN  2 , total integrated cost =  208.96595381043926
RUN  3 , total integrated cost =  100.33196111026527
RUN  4 , total integrated cost =  49.25978056695863
RUN  5 , total integrated cost =  46.7362846288914
RUN  6 , total integrated cost =  43.955874065773386
RUN  7 , total integrated cost =  40.08872858155644
RUN  8 , total integrated cost =  37.44356312274883
RUN  9 , total integrated cost =  35.783843214120644
RUN  10 , total integrated cost =  34.91521608842192
RUN  11 , total integrated cost =  34.89863107073908
RUN  12 , total integrated cost =  34.79665682252719
RUN  13 , total integrated cost =  34.73680011882326
RUN  14 , total integrated cost =  34.69571004783956
RUN  15 , total integrated cost =  34.621625945684706
RUN  16 

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  33.527144560289784
State only changes marginally.
Control only changes marginally.
RUN  47 , total integrated cost =  33.52714456021137
Improved over  47  iterations in  14.929705783724785  seconds by  99.34225547908362  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447290925615 -56.62447253255006
weight =  1520.347138136236
set cost params:  1.0 1520.347138136236 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5070.7251927478765
Gradient descend method:  None
RUN  1 , total integrated cost =  4607.343692579996
RUN  2 , total integrated cost =  4606.942887880815
RUN  3 , total integrated cost =  4606.891162668009
RUN  4 , total integrated cost =  4606.865769926869
RUN  5 , total integrated cost =  4606.830365082819
RUN  6 , total integrated cost =  4606.680541273461
RUN  7 , total integrated cost =  4580.984538892031
RUN  8 , total integrated cost =  4580.98453889203


ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  4580.98453889203
Control only changes marginally.
RUN  9 , total integrated cost =  4580.98453889203
Improved over  9  iterations in  0.5123088788241148  seconds by  9.658197501143832  percent.
Problem in initial value trasfer:  Vmean_exc -56.62877603751035 -56.62871539784095
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  794.6129979848286
RUN  2 , total integrated cost =  388.02512020586994
RUN  3 , total integrated cost =  71.90934112750469
RUN  4 , total integrated cost =  68.62109965871412
RUN  5 , total integrated cost =  65.33848807439037
RUN  6 , total integrated cost =  62.017483489401315
RUN  7 , total integrated cost =  58.73105429664529
RUN  8 , total integrated cost =  55.64025404131153
RUN  9 , total integrated cost =  51.979262397133304
RUN  10 , total

...

RUN  2 , total integrated cost =  23532.63614302563
RUN  3 , total integrated cost =  23532.63614302563
Control only changes marginally.
RUN  3 , total integrated cost =  23532.63614302563
Improved over  3  iterations in  0.17852861061692238  seconds by  2.9046987037872896e-10  percent.
Problem in initial value trasfer:  Vmean_exc -73.65794409599617 -73.6579441840707
weight =  10.000000000029047
set cost params:  1.0 10.000000000029047 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.63614302563
Gradient descend method:  None
RUN  1 , total integrated cost =  23532.63614302563


ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1 , total integrated cost =  23532.63614302563
Improved over  1  iterations in  0.09834533929824829  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.65794409599617 -73.6579441840707
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  33290.047977386734
RUN  2 , total integrated cost =  33290.04797738673
RUN  3 , total integrated cost =  33290.04797738672


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  33290.04797738672
Control only changes marginally.
RUN  4 , total integrated cost =  33290.04797738672
Improved over  4  iterations in  0.2675304841250181  seconds by  1.048208352472102e-05  percent.
Problem in initial value trasfer:  Vmean_exc -68.82423325167102 -68.8242522248443
weight =  10.000001048208462
set cost params:  1.0 10.000001048208462 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.047977387076
Gradient descend method:  None
RUN  1 , total integrated cost =  33290.047977387076
Control only changes marginally.
RUN  1 , total integrated cost =  33290.047977387076
Improved over  1  iterations in  0.10626736655831337  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -68.82423325167102 -68.8242522248443


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  8231.907221276197
RUN  6 , total integrated cost =  8231.907221276195
RUN  7 , total integrated cost =  8231.907221276193
RUN  8 , total integrated cost =  8231.907221276193
Control only changes marginally.
RUN  8 , total integrated cost =  8231.907221276193
Improved over  8  iterations in  0.3387568760663271  seconds by  0.7421873145890174  percent.
Problem in initial value trasfer:  Vmean_exc -75.18626566871573 -75.18626665459846
weight =  10.00000000023317
set cost params:  1.0 10.00000000023317 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8231.907221276193
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  8231.907221276193
Control only changes marginally.
RUN  1 , total integrated cost =  8231.907221276193
Improved over  1  iterations in  0.08819575980305672  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -75.18626566871573 -75.18626665459846
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
[0, 5, 10, 15, 20, 30, 35, 40] []
closest index  40
set cost params:  1.0 10.0 0.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20688.257887262374
Gradient descend method:  None
RUN  1 , total integrated cost =  20627.990473513702
RUN  2 , total integrated cost =  20627.9071067425
RUN  3 , total integrated cost =  20627.90706726546
RUN  4 , total integrated cost =  20627.90706725

...

RUN  11 , total integrated cost =  19365.88746254103
RUN  12 , total integrated cost =  19365.88746254097
RUN  13 , total integrated cost =  19365.887462540948
RUN  14 , total integrated cost =  19365.887462540944
RUN  15 , total integrated cost =  19365.887462540944
Control only changes marginally.
RUN  15 , total integrated cost =  19365.887462540944
Improved over  15  iterations in  0.620262149721384  seconds by  2.5202304512621794  percent.
Problem in initial value trasfer:  Vmean_exc -56.695132330992884 -56.69513398862267
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
found solution for  85
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
[0, 5, 10, 15, 20, 30, 35, 40, 50, 55, 60, 70, 75, 80, 90, 100, 105, 110, 120, 130, 140, 85] [80]
closest index  110
set cost params:  1.0 10.0

ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  355.90775758811236
RUN  14 , total integrated cost =  355.90775758811225
RUN  15 , total integrated cost =  355.90775758811225
Control only changes marginally.
RUN  15 , total integrated cost =  355.90775758811225
Improved over  15  iterations in  0.6908192317932844  seconds by  35.430233818911105  percent.
Problem in initial value trasfer:  Vmean_exc -56.70140805965836 -56.701408091089654
weight =  677.94089867897
set cost params:  1.0 677.94089867897 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23960.450001353714
Gradient descend method:  None
RUN  1 , total integrated cost =  23558.846717125536
RUN  2 , total integrated cost =  23554.787402370584
RUN  3 , total integrated cost =  23554.78561005346
RUN  4 , total integrated cost =  23554.7855549474
RUN  5 , total integrated cost =  23554.785554037222
RUN  6 , total integrated cost =  23554.785554030717


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  23554.7855540307
RUN  8 , total integrated cost =  23554.785554030685
RUN  9 , total integrated cost =  23554.785554030677
RUN  10 , total integrated cost =  23554.78555403067
RUN  11 , total integrated cost =  23554.785554030663
RUN  12 , total integrated cost =  23554.785554030663
Control only changes marginally.
RUN  12 , total integrated cost =  23554.785554030663
Improved over  12  iterations in  0.32524807192385197  seconds by  1.6930585498190993  percent.
Problem in initial value trasfer:  Vmean_exc -56.70139886607887 -56.70139922831183
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
found solution for  115
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
found solution for  125
-------  130 0.6000000000000003 0.8500000000000005
-------  1

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  1690.6996634153118
set cost params:  1.0 1690.6996634153118 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5074.6004499212895
Gradient descend method:  None
RUN  1 , total integrated cost =  5074.29500129625
RUN  2 , total integrated cost =  5074.294337973072
RUN  3 , total integrated cost =  5074.2943379687185


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  5074.29433796869
RUN  5 , total integrated cost =  5074.294337968686
RUN  6 , total integrated cost =  5074.294337968681
RUN  7 , total integrated cost =  5074.294337968681
Control only changes marginally.
RUN  7 , total integrated cost =  5074.294337968681
Improved over  7  iterations in  0.40627409145236015  seconds by  0.0060322375254742155  percent.
Problem in initial value trasfer:  Vmean_exc -56.629176342482616 -56.62911352662408
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  3635.77923007471
set cost params:  1.0 3635.77923007471 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12973.342860962672
Gradient descend method:  None
RUN  1 , total integrated cost =  12970.91480929144
RUN  2 , total integrated cost =  12970.88753125927
RUN  3 , total integrated cost =  12970.886776291343
RUN  4 , total integrated cost =  12970.886

ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  12970.886765602361
RUN  10 , total integrated cost =  12970.886765602358
State only changes marginally.
RUN  11 , total integrated cost =  12970.886765602358
Control only changes marginally.
RUN  11 , total integrated cost =  12970.886765602358
Improved over  11  iterations in  0.5468808598816395  seconds by  0.018931861946740014  percent.
Problem in initial value trasfer:  Vmean_exc -56.66980833173944 -56.66982869898958
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  648.3432433152989
set cost params:  1.0 648.3432433152989 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8213.301201851531
Gradient descend method:  None
RUN  1 , total integrated cost =  8213.249441487907
RUN  2 , total integrated cost =  8213.243981947902
RUN  3 , total integrated cost =  8213.242745184065
RUN  4 , total integrated cost =  8213.242363219408
RUN  5 

...

RUN  1 , total integrated cost =  20615.723577302284
Control only changes marginally.
RUN  1 , total integrated cost =  20615.723577302284
Improved over  1  iterations in  0.07497468031942844  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69638238772046 -56.696383829602325
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
no convergence
weight =  838.3508208085366
set cost params:  1.0 838.3508208085366 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20047.202448524462
Gradient descend method:  None
RUN  1 , total integrated cost =  20047.202448524462
Control only changes marginally.
RUN  1 , total integrated cost =  20047.202448524462
Improved over  1  iterations in  0.07759913802146912  seconds by  0.0  percent.
Problem in ini

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13014.507357741055
RUN  2 , total integrated cost =  13014.507357741055
Control only changes marginally.
RUN  2 , total integrated cost =  13014.507357741055
Improved over  2  iterations in  0.1300803404301405  seconds by  4.263256414560601e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.66980669549727 -56.669827100880475
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  13014.507357741059
Control only changes marginally.
RUN  1 , total integrated cost =  13014.507357741059
Improved over  1  iterations in  0.07978910580277443  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.66980669549727 -56.669827100880475
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.45

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  211.305757203619
Gradient descend method:  None
RUN  1 , total integrated cost =  25.90007694367825
RUN  2 , total integrated cost =  25.885902084016305
RUN  3 , total integrated cost =  25.88500356481579
RUN  4 , total integrated cost =  25.884766493054816
RUN  5 , total integrated cost =  25.884209010061735
RUN  6 , total integrated cost =  25.881912950863736
RUN  7 , total integrated cost =  25.879533932763604
RUN  8 , total integrated cost =  25.879382850111273
RUN  9 , total integrated cost =  25.879061933979813
RUN  10 , total integrated cost =  25.87857393454917
RUN  11 , total integrated cost =  25.87545308993137
RUN  12 , total integrated cost =  25.870259254418343
RUN  13 , total integrated cost =  25.870177034458244
RUN  14 , total integrated cost =  25.870019284653456
RUN  15 , total integrated cost =  25.869385020695887
RU

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  58 , total integrated cost =  25.862690736801884
Improved over  58  iterations in  4.3421121165156364  seconds by  87.7605366370212  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447063133888 -56.62447038764511
weight =  100
set cost params:  1.0 100.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  330.5836185352296
Gradient descend method:  HS
RUN  1 , total integrated cost =  322.14827510639867
RUN  2 , total integrated cost =  321.3934541802533
RUN  3 , total integrated cost =  311.4960341712016
RUN  4 , total integrated cost =  309.965005009921
RUN  5 , total integrated cost =  304.97394294173245
RUN  6 , total integrated cost =  302.3059422514904
RUN  7 , total integrated cost =  298.5267179134578
RUN  8 , total integrated cost =  298.5267179134574
RUN  9 , total integrated cost =  295.23029395497366
RUN  10 , total integrated cost =  295.0455890461244
RUN  11 , total integrated cost =  291.4654532714

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  248 , total integrated cost =  193.23121794241663
Improved over  248  iterations in  24.660119522362947  seconds by  41.548459418952014  percent.
Problem in initial value trasfer:  Vmean_exc -56.624483784124195 -56.62448298132134
weight =  2636.922527465892
set cost params:  1.0 2636.922527465892 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5082.017224591562
Gradient descend method:  None
RUN  1 , total integrated cost =  5013.61708336626
RUN  2 , total integrated cost =  5002.289911622725
RUN  3 , total integrated cost =  4979.523169332194
RUN  4 , total integrated cost =  4975.023616815568
RUN  5 , total integrated cost =  4886.425935505947
RUN  6 , total integrated cost =  4843.930942283712
RUN  7 , total integrated cost =  4843.434740147745
RUN  8 , total integrated cost =  4841.969483739279
RUN  9 , total integrated cost =  4841.2244245102975
RUN  10 , total integrated cost =  4840.610474388173
RUN  11 , total in

...

Control only changes marginally.
RUN  199 , total integrated cost =  8206.27890845016
Improved over  199  iterations in  10.812231592833996  seconds by  65.07622408929808  percent.
Problem in initial value trasfer:  Vmean_exc -56.700662968179145 -56.700663858729364
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  359.35453148856686
Gradient descend method:  None
RUN  1 , total integrated cost =  93.72436190826632
RUN  2 , total integrated cost =  93.35957362120303
RUN  3 , total integrated cost =  93.16244987017137
RUN  4 , total integrated cost =  93.05797854328321
RUN  5 , total integrated cost =  93.01992581111631
RUN  6 , total integrated cost =  92.9578443893427
RUN  7 , total integrated cost =  92.93130418527194
RUN  8 , total integrated cost =  92.89733022968905
RUN  9 , total integrated cost =  92.88974221622823
RUN  10 , total integrated cost =  92.86412580833192
RUN  11 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  78 , total integrated cost =  717.218647828825
Improved over  78  iterations in  5.707516340538859  seconds by  71.62079172207876  percent.
Problem in initial value trasfer:  Vmean_exc -56.70407589193722 -56.70405206113213
weight =  4640.548510716203
set cost params:  1.0 4640.548510716203 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29580.610254480427
Gradient descend method:  None
RUN  1 , total integrated cost =  21635.985969398687
RUN  2 , total integrated cost =  20318.17383690787
RUN  3 , total integrated cost =  19005.012682344764
RUN  4 , total integrated cost =  18418.901681520052
RUN  5 , total integrated cost =  17896.803233103168
RUN  6 , total integrated cost =  17606.461513037233
RUN  7 , total integrated cost =  17322.054897015372
RUN  8 , total integrated cost =  17178.119700481235
RUN  9 , total integrated cost =  17048.466606576934
RUN  10 , total integrated cost =  16988.018340996157
RUN  11 , total

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  32 , total integrated cost =  16268.386342567528
Improved over  32  iterations in  2.178462967276573  seconds by  45.00320918800708  percent.
Problem in initial value trasfer:  Vmean_exc -56.7035629888963 -56.70356127735979


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2776.441217442964
set cost params:  1.0 2776.441217442964 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5093.1721272438845
Gradient descend method:  None
RUN  1 , total integrated cost =  5093.101569105319
RUN  2 , total integrated cost =  5093.101488201135
RUN  3 , total integrated cost =  5093.1014878564265
RUN  4 , total integrated cost =  5093.101487856418
RUN  5 , total integrated cost =  5093.101487856416


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5093.101487856416
Control only changes marginally.
RUN  6 , total integrated cost =  5093.101487856416
Improved over  6  iterations in  0.8778513390570879  seconds by  0.0013869428659489813  percent.
Problem in initial value trasfer:  Vmean_exc -56.624802231881574 -56.624796773639645
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  6049.870179382591
set cost params:  1.0 6049.870179382591 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13015.024388576776
Gradient descend method:  None
RUN  1 , total integrated cost =  13015.01593131583
RUN  2 , total integrated cost =  13015.015915962469
RUN  3 , total integrated cost =  13015.01591577384
RUN  4 , total integrated cost =  13015.015915772656
RUN  5 , total integrated cost =  13015.01591577264
RUN  6 , total integrated cost =  13015.01591577263


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  13015.01591577263
Control only changes marginally.
RUN  7 , total integrated cost =  13015.01591577263
Improved over  7  iterations in  1.0221743639558554  seconds by  6.510017878724739e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67053684347293 -56.67053953200231
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1040.859643632126
set cost params:  1.0 1040.859643632126 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8223.384301906994
Gradient descend method:  None
RUN  1 , total integrated cost =  8223.373190649314
RUN  2 , total integrated cost =  8223.373190649305
RUN  3 , total integrated cost =  8223.373190649301
RUN  4 , total integrated cost =  8223.373190649301
Control only changes marginally.
RUN  4 , total integrated cost =  8223.373190649301
Improved over  4  iterations in  0.5971129667013884  seconds by  0.00013511782113084791  percent.
-------  45 0.5000000000000002 0.5

...

RUN  1 , total integrated cost =  20061.13836226856
Control only changes marginally.
RUN  1 , total integrated cost =  20061.13836226856
Improved over  1  iterations in  0.19971302896738052  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.69517068115736 -56.695171013702165
-------  75 0.5750000000000002 0.6750000000000004
no convergence
weight =  30468.585758720434
set cost params:  1.0 30468.585758720434 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  34494.6968440686
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  34494.6968440686
Control only changes marginally.
RUN  1 , total integrated cost =  34494.6968440686
Improved over  1  iterations in  0.2081518080085516  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70311267924185 -56.703113263518546
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
no convergence
weight =  9554.234309994017
set cost params:  1.0 9554.234309994017 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33286.567507156076
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  33286.567507156076
Control only changes marginally.
RUN  1 , total integrated cost =  33286.567507156076
Improved over  1  iterations in  0.20526813343167305  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70356371476446 -56.703561977258666
--------------- 8
[[True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.575000000000

In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  47.74221812391439
Gradient descend method:  None
RUN  1 , total integrated cost =  5.464633252415316
RUN  2 , total integrated cost =  5.4339990511326395
RUN  3 , total integrated cost =  5.43352745071552
RUN  4 , total integrated cost =  5.433525265339015
RUN  5 , total integrated cost =  5.433525257978253
RUN  6 , total integrated cost =  5.433525257914094
RUN  7 , total integrated cost =  5.433525257913553


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  5.433525257913521
RUN  9 , total integrated cost =  5.433525257913505
RUN  10 , total integrated cost =  5.433525257913505
Control only changes marginally.
RUN  10 , total integrated cost =  5.433525257913505
Improved over  10  iterations in  0.26152139715850353  seconds by  88.6190347423514  percent.
Problem in initial value trasfer:  Vmean_exc -56.62462913071813 -56.62462456051956
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  51.81938359775803
Gradient descend method:  None
RUN  1 , total integrated cost =  2.4596836393596204
RUN  2 , total integrated cost =  2.4531581245732155
RUN  3 , total integrated cost =  2.4531094464587886
RUN  4 , total integrated cost =  2.4530818169839743
RUN  5 , total integrated cost =  2.4527398738330795
RUN  6 , total integrated cost =  2.451385901577454
RUN  7 , total integrated cost =  2.451242198420716
RUN  8 , t

ERROR:root:Problem in initial value trasfer


RUN  100 , total integrated cost =  2.293321273183389
Control only changes marginally.
RUN  107 , total integrated cost =  2.293321273183312
Improved over  107  iterations in  2.4531901199370623  seconds by  95.57439491950551  percent.
Problem in initial value trasfer:  Vmean_exc -56.670684117795325 -56.67068382340939
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  26.28428066302708
Gradient descend method:  None


...

Control only changes marginally.
RUN  89 , total integrated cost =  23.613489215713273
Improved over  89  iterations in  1.8982955198734999  seconds by  55.038298958302704  percent.
Problem in initial value trasfer:  Vmean_exc -56.677021047517364 -56.67703493104884
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  99.78380010131917
Gradient descend method:  None
RUN  1 , total integrated cost =  12.096540287361659
RUN  2 , total integrated cost =  12.05539968515911
RUN  3 , total integrated cost =  12.053453644560703
RUN  4 , total integrated cost =  12.048141238005941
RUN  5 , total integrated cost =  12.046792586058455
RUN  6 , total integrated cost =  12.034924899428981
RUN  7 , total integrated cost =  12.019569761566611
RUN  8 , total integrated cost =  12.018859107507952
RUN  9 , total integrated cost =  11.864250456930812


ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  11.844170948657315
RUN  11 , total integrated cost =  11.844170948657288
RUN  12 , total integrated cost =  11.844170948657288
Control only changes marginally.
RUN  12 , total integrated cost =  11.844170948657288
Improved over  12  iterations in  0.31713163293898106  seconds by  88.13016648330604  percent.
Problem in initial value trasfer:  Vmean_exc -56.70066955081464 -56.70067012220561
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  308.9296006641629
Gradient descend method:  None
RUN  1 , total integrated cost =  4.041108078185289
RUN  2 , total integrated cost =  4.037882923815128
RUN  3 , total integrated cost =  4.0367220037002935
RUN  4 , total integrated cost =  4.032798292294316
RUN  5 , total integrated cost =  4.031841854206201
RUN  6 , total integrated cost =  4.030880486863508
RUN  7 , total integrated cost =  4.027540952753124
RUN  8

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  28 , total integrated cost =  4.0094236876637
Improved over  28  iterations in  0.6170803513377905  seconds by  98.70215619382412  percent.
Problem in initial value trasfer:  Vmean_exc -56.703556931531416 -56.703555574886025


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.433525257913505
Gradient descend method:  None
RUN  1 , total integrated cost =  5.433525257913505
Control only changes marginally.
RUN  1 , total integrated cost =  5.433525257913505
Improved over  1  iterations in  0.07005770318210125  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62462913071813 -56.62462456051956
-------  15 0.4500000

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  9.956542371212713
Control only changes marginally.
RUN  1 , total integrated cost =  9.956542371212713
Improved over  1  iterations in  0.0677160918712616  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.640411159217095 -56.64040082127102
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7.065863198619762
Gradient descend method:  None
RUN  1 , total integrated cost =  7.065863198619762
Control only changes marginally.
RUN  1 , total integrated cost =  7.065863198619762
Improved over  1  iterations in  0.08023927360773087  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.696424991573494 -56.69642507785499
-------  65 0.5000000000000002 0.6500000000000004
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  11.1184350789722

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.245913403623962
Control only changes marginally.
RUN  1 , total integrated cost =  1.245913403623962
Improved over  1  iterations in  0.10814180038869381  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.7031142058964 -56.70311466849832
-------  85 0.47500000000000014 0.7250000000000004
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  22.031647475165776
Gradient descend method:  None
RUN  1 , total integrated cost =  22.031647475165776
Control only changes marginally.
RUN  1 , total integrated cost =  22.031647475165776
Improved over  1  iterations in  0.08903488889336586  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67980013630912 -56.679804072244856
-------  95 0.5250000000000001 0.7500000000000004
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.89445697158

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  8.894456971585095
Control only changes marginally.
RUN  1 , total integrated cost =  8.894456971585095
Improved over  1  iterations in  0.09583056345582008  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70140986395801 -56.701409792237044
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23.613489215713273
Gradient descend method:  None
RUN  1 , total integrated cost =  23.613489215713273
Control only changes marginally.
RUN  1 , total integrated cost =  23.613489215713273
Improved over  1  iterations in  0.06658692844212055  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.677021047517364 -56.67703493104884
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 1.0 0.0
interpolate

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  4.0094236876637
Control only changes marginally.
RUN  1 , total integrated cost =  4.0094236876637
Improved over  1  iterations in  0.06442820653319359  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.703556931531416 -56.703555574886025
--------------- 11
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [True, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.575

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
